In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Pick a filter type, then drag its cutoff $f_C$ and its $Q$. The red curve is
the filter's magnitude response in decibels, on a log-frequency axis, and the
gold dots mark where it crosses the dashed $-6$ dB line. For the low-pass and
high-pass filters, that crossing is the cutoff $f_C$, and raising $Q$ grows a
resonant peak next to it. For the band-pass and band-stop filters, $f_C$ is
the center, and the readout measures $Q$ from the two edges $f_L$ and $f_H$.
The audio card plays the source through the filter.

In [ ]:
# hide
# autorun
KIND0, FC0, Q0, SOURCE0 = "Low-pass", 800.0, 4.0, "Sawtooth (110 Hz)"

SR = 44100
F = np.geomspace(20.0, 20000.0, 2000)
Z1 = np.exp(-2j * np.pi * F / SR)          # z^-1 on the unit circle
DB_LO, DB_HI = -60.0, 30.0


def cookbook(kind, fc, q, SR=SR):
    """(b, a) of a second-order filter, straight from the Audio EQ Cookbook."""
    w0 = 2 * np.pi * fc / SR
    c, alpha = np.cos(w0), np.sin(w0) / (2 * q)
    if kind == "Low-pass":
        b = [(1 - c) / 2, 1 - c, (1 - c) / 2]
    elif kind == "High-pass":
        b = [(1 + c) / 2, -(1 + c), (1 + c) / 2]
    elif kind == "Band-pass":
        b = [alpha, 0.0, -alpha]
    else:
        b = [1.0, -2 * c, 1.0]
    a = [1 + alpha, -2 * c, 1 - alpha]
    return np.array(b) / a[0], np.array(a) / a[0]


def response_db(b, a, Z1=Z1):
    num = b[0] + b[1] * Z1 + b[2] * Z1 ** 2
    den = a[0] + a[1] * Z1 + a[2] * Z1 ** 2
    return 20 * np.log10(np.maximum(np.abs(num / den), 1e-6))


def edges(db, F=F):
    """Frequencies where the response crosses -6 dB."""
    s = np.sign(db + 6.0)
    i = np.nonzero(s[:-1] != s[1:])[0]
    t = (-6.0 - db[i]) / (db[i + 1] - db[i])
    return np.exp(np.log(F[i]) + t * (np.log(F[i + 1]) - np.log(F[i])))


def biquad(kind, fc, q, SR=SR, cookbook=cookbook, response_db=response_db,
           edges=edges):
    """The cookbook filter, set up with the chapter's definitions. For the
    low-pass and high-pass filters, fc is the cutoff where the response
    crosses -6 dB (the cookbook's own corner frequency sits elsewhere). For
    the band filters, fc is the center and q is the center over the bandwidth
    between the -6 dB edges (the cookbook measures bandwidth at -3 dB)."""
    if kind in ("Low-pass", "High-pass"):
        # the cookbook filters are analog prototypes, warped by the bilinear
        # transform; in the prototype, the -6 dB point sits at u = sqrt(v)
        # times the corner, where v^2 + (1/q^2 - 2) v - 3 = 0
        p = 2 - 1 / q ** 2
        u = np.sqrt((p + np.sqrt(p ** 2 + 12)) / 2)
        u = 1 / u if kind == "Low-pass" else u
        return cookbook(kind, SR / np.pi * np.arctan(np.tan(np.pi * fc / SR) * u), q)
    # a -6 dB bandwidth is sqrt(3) times the -3 dB one in the prototype; the
    # warping then stretches it a little, so refine q until it measures right
    q_cb = q * np.sqrt(3) if kind == "Band-pass" else q / np.sqrt(3)
    for _ in range(4):
        e = edges(response_db(*cookbook(kind, fc, q_cb)))
        if len(e) < 2:
            break
        q_cb *= q * (e[-1] - e[0]) / fc
    return cookbook(kind, fc, q_cb)


def impulse_response(b, a, n_max=2 * SR):
    """Run the filter on a unit impulse until it has decayed by 80 dB."""
    h = np.zeros(n_max)
    y1 = y2 = 0.0
    quiet = 0
    for n in range(n_max):
        x0 = 1.0 if n == 0 else 0.0
        x1 = 1.0 if n == 1 else 0.0
        x2 = 1.0 if n == 2 else 0.0
        y0 = b[0] * x0 + b[1] * x1 + b[2] * x2 - a[1] * y1 - a[2] * y2
        h[n] = y0
        y2, y1 = y1, y0
        quiet = quiet + 1 if abs(y0) < 1e-4 else 0
        if quiet > 64:
            return h[:n + 1]
    return h


def convolve(x, h):
    """Fast convolution (the convolution theorem), trimmed to len(x)."""
    L = 1 << int(np.ceil(np.log2(len(x) + len(h) - 1)))
    return np.fft.irfft(np.fft.rfft(x, L) * np.fft.rfft(h, L), L)[:len(x)]


# two seconds of each source: a band-limited sawtooth (one 400-sample period
# summed from its harmonics, then repeated) and white noise
_P = 400                                   # 44,100 / 400 = 110.25 Hz
_n = np.arange(_P)
_period = sum(np.sin(2 * np.pi * k * _n / _P) / k for k in range(1, _P // 2))
SOURCES = {
    "Sawtooth (110 Hz)": np.tile(_period / np.abs(_period).max(), 2 * SR // _P),
    "White noise": np.random.default_rng(0).uniform(-1.0, 1.0, 2 * SR),
}


def figure():
    b, a = biquad(KIND0, FC0, Q0)
    db = response_db(b, a)
    e = edges(db)
    fig = go.Figure()
    fig.add_scatter(x=[20, 20000], y=[-6, -6], mode="lines",
                    line=dict(color=IRON, width=1.2, dash="dash"))
    fig.add_scatter(x=[FC0, FC0], y=[DB_LO, DB_HI], mode="lines",
                    line=dict(color=STEEL, width=1.2, dash="dot"))
    fig.add_scatter(x=F, y=db, mode="lines", line=dict(color=RED, width=2.4))
    fig.add_scatter(x=e, y=np.full(len(e), -6.0), mode="markers",
                    marker=dict(color=GOLD, size=10, line=dict(color=IRON, width=1)))
    fig.update_xaxes(type="log", range=[np.log10(20), np.log10(20000)],
                     title_text="Frequency (Hz)", fixedrange=True)
    fig.update_yaxes(range=[DB_LO, DB_HI], title_text="Magnitude (dB)",
                     fixedrange=True)
    return fig


def controls(fig):
    kind = widgets.Dropdown(description="Filter type", value=KIND0,
                            options=["Low-pass", "High-pass", "Band-pass",
                                     "Band-stop"])
    fc = widgets.FloatLogSlider(description="Cutoff $f_C$ (Hz)", base=10,
                                min=np.log10(FC0) - 1.21, max=4.0, step=0.01, value=FC0,
                                readout_format=".0f")
    q = widgets.FloatLogSlider(description="$Q$", base=10, min=np.log10(Q0) - 0.91,
                               max=np.log10(20), step=0.01, value=Q0,
                               readout_format=".2f")
    source = widgets.Dropdown(description="Audio source", value=SOURCE0,
                              options=list(SOURCES))
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(kind, fc, q, biquad=biquad, response_db=response_db, edges=edges,
               F=F, readout=readout):
        b, a = biquad(kind, fc, q)
        db = response_db(b, a)
        e = edges(db)
        with fig.batch_update():
            fig.data[1].x = [fc, fc]
            fig.data[2].y = db
            fig.data[3].x, fig.data[3].y = e, np.full(len(e), -6.0)
        if kind in ("Band-pass", "Band-stop") and len(e) >= 2:
            fl, fh = e[0], e[-1]
            text = (f"&minus;6 dB edges <i>f</i><sub>L</sub> = {fl:,.0f} Hz and "
                    f"<i>f</i><sub>H</sub> = {fh:,.0f} Hz &nbsp;·&nbsp; bandwidth "
                    f"{fh - fl:,.0f} Hz &nbsp;·&nbsp; <i>f</i><sub>C</sub> / "
                    f"(<i>f</i><sub>H</sub> &minus; <i>f</i><sub>L</sub>) = "
                    f"{fc / (fh - fl):.2f}")
        elif len(e):
            text = (f"cutoff: the response crosses &minus;6 dB at "
                    f"<i>f</i><sub>C</sub> = {e[0]:,.0f} Hz")
            peak = db.max()
            if peak > 0.1:
                text += (f" &nbsp;·&nbsp; resonant peak +{peak:.1f} dB at "
                         f"{F[db.argmax()]:,.0f} Hz")
        else:
            text = "no &minus;6 dB crossing in the audible range"
        readout.value = f"<span style='font-size:0.9em'>{text}</span>"

    widgets.interactive_output(update, {"kind": kind, "fc": fc, "q": q})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(SOURCES=SOURCES, biquad=biquad, impulse_response=impulse_response,
               convolve=convolve):
        b, a = biquad(kind.value, fc.value, q.value)
        y = convolve(SOURCES[source.value], impulse_response(b, a))
        y = 0.5 * y / max(np.abs(y).max(), 1e-9)   # -6 dBFS: resonance can boost the level
        y[:441] *= np.linspace(0, 1, 441)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (kind, fc, q, source):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([kind, fc, q, source, readout, out, gate])

icm_plotly.show(figure, controls)